# 01 · Black-box optimization quickstart (`nevergrad` / CMA-ES / `pymoo`)

**What this notebook is.** A copy-paste template for driving the origami
folding simulation from an external black-box optimizer. The library supplies
the *search space* (`SearchSpace` / `DesignSpace`) and the *objective*
(`OrigamiEvaluator`); **it ships no optimizer**. `nevergrad`, `cma` and
`pymoo` run the search; this notebook only glues them to the simulation.

**What you will do**

1. build a 4-variable design space over a tiny 2×2 Miura pattern;
2. score a genome with the real physics evaluator and inspect descriptors;
3. seed an initial population (Latin hypercube / mixed with provenance);
4. run `nevergrad`, CMA-ES and `pymoo` against one `genome -> loss` interface;
5. checkpoint the best design as a normal crease-pattern JSON file.

> ### Why the loops default to a *surrogate* objective
>
> One MJX-JAX physics evaluation of this 6-face model takes **~12 s on a CPU
> runtime**, and the 4×4 reference pattern takes minutes per step. The notebook
> therefore wires the optimizers to a cheap analytic stand-in with the **same
> contract** (`genome -> float to minimize`) and re-scores the winner with the
> real `OrigamiEvaluator` at the end. Set `OBJECTIVE_MODE = "physics"` in the
> config cell to optimize the actual forward displacement — do that on a GPU
> runtime with a real budget.
>
> Only the *objective* changes between the two modes; the space, bounds,
> `repair`, logging and checkpoints are identical, and each loop is written
> once.


## Install

Run this once per runtime. The cell installs the package straight from the main
repository, [https://github.com/ihsan31415/origami-engine-mjx](https://github.com/ihsan31415/origami-engine-mjx),
at a fixed commit, so the notebook keeps working even as the project moves on.
Replace `@957cc57` with `@main` to follow the latest code, or install
`origami-engine-mjx` from PyPI once you have it locally.

In [ ]:
%pip install -q "origami-engine-mjx[jax] @ git+https://github.com/ihsan31415/origami-engine-mjx.git@957cc57"
%pip install -q nevergrad cma pymoo

## 0. Configure the run

Everything tunable lives in this cell.


In [ ]:
import numpy as np

from origami_engine_mjx import Crease
from origami_engine_mjx.integrations import (
    DesignSpace,
    EvalRecord,
    OrigamiEvaluator,
    RunLogger,
    lhs_population,
    load_best,
    mixed_population,
    population_stats,
    save_best,
    save_genome,
)
from origami_engine_mjx.integrations.adapters import space_spec

OBJECTIVE_MODE = "surrogate"  # "surrogate" (CPU-fast) or "physics" (real scores)
USE_DEFAULT_PATTERN = False  # True -> packaged 4x4 Miura, 15 variables
FAST_DEMO = True  # False -> larger optimizer budgets

# Simulated seconds per evaluation. Keep these small on CPU.
EVAL_DURATION = 0.004
EVAL_FOLD = 0.003

BUDGET = 8 if FAST_DEMO else 128
POP_SIZE = 4

print("numpy", np.__version__)

## 1. Define the search space

`DesignSpace.angles_only(pattern)` keeps the crease topology fixed and varies
only the actuated fold angles — the same semantics as `SearchSpace`, plus the
normalized `[-1, 1]` view and `base_vector()` used below.

The 2×2 Miura pattern has 6 creases and **4 actuated hinges**; the packaged 4×4
pattern has 24 creases and 15 actuated hinges.


In [ ]:
if USE_DEFAULT_PATTERN:
    space = DesignSpace.from_default_pattern()
else:
    pattern = [
        Crease((0.1, 0.1), (0.1, 0.5), 0.8),
        Crease((0.1, 0.5), (0.5, 0.5), -0.8),
        Crease((0.5, 0.5), (0.5, 0.1), 0.8),
        Crease((0.5, 0.1), (0.1, 0.1), -0.8),
        Crease((0.1, 0.1), (0.5, 0.5), -0.8),
        Crease((0.1, 0.5), (0.5, 0.1), 0.8),
    ]
    space = DesignSpace.angles_only(pattern, name="miura-2x2")

spec = space_spec(space)
print(f"space      : {space.name!r}")
print(f"dim        : {spec['dim']}  (actuated hinges)")
print(f"n_creases  : {spec['n_creases']}")
print(f"lower      : {np.round(spec['lower'], 3).tolist()}")
print(f"upper      : {np.round(spec['upper'], 3).tolist()}")
print(f"names[:4]  : {spec['names'][:4]}")

In [ ]:
# The genome lives in angle space; the normalized view is [-1, 1] and gives
# optimizer steps a scale-free coordinate system.
genome = space.base_vector()  # reproduces the input pattern exactly
z = space.to_normalized(genome)
print("base genome      :", np.round(genome, 4).tolist())
print("normalized view  :", np.round(z, 4).tolist())
assert np.allclose(space.from_normalized(z), genome)

print("sample(0)        :", np.round(space.sample(0), 4).tolist())
print("repaired NaN vec :", np.round(space.repair(np.full(space.dim, np.nan)), 4).tolist())

## 2. The real objective (`OrigamiEvaluator`)

`OrigamiEvaluator(space, backend="jax", ...)` wraps one full `OrigamiEngine.run`
+ `compute_forward_displacement`. Fitness is **always higher-is-better**: with
`maximize=True` (default) it is the raw +x centroid displacement in metres.

* `evaluate(genome) -> EvalRecord` — one simulation plus behavior descriptors.
* `evaluate_batch(X, return_details=False) -> BatchResult` — the fitness-only
  fast path. **Honest note:** both backends still loop per individual today, so
  this is not a throughput win; it only skips descriptor bookkeeping.
* Failures never crash a batch: bad genomes get `status="invalid"` and a large
  negative penalty (`error_policy="penalty"`).


In [ ]:
evaluator = OrigamiEvaluator(
    space,
    backend="jax",  # MJX on JAX; the Warp backend is not wrapped by env
    duration=EVAL_DURATION,
    fold_duration=EVAL_FOLD,
    vibrate=False,  # deterministic fold-only score
    maximize=True,  # fitness = raw forward displacement
    error_policy="penalty",
)

record = evaluator.evaluate(space.base_vector())
print(f"fitness={record.fitness:+.6f} m   raw={record.raw:+.6f} m")
print(f"status={record.status!r}  path={record.path!r}  time={record.time:.1f} s")
print("descriptors:", {k: round(v, 5) for k, v in record.descriptors.items()})

In [ ]:
# A two-genome batch shows the BatchResult surface. Keep batches tiny on CPU:
# the backend loops per individual, so two genomes cost two simulations.
X = space.sample(1, size=2)
batch = evaluator.evaluate_batch(X, return_details=False)
print("fitness :", np.round(batch.fitness, 6).tolist())
print("status  :", batch.status)
print("paths   :", batch.paths)
print("best    : index", batch.best_index(), "->", round(float(batch.fitness.max()), 6), "m")

## 3. Seed the initial population

Population helpers are pure NumPy and *do not* run the simulation, so they are
free even in physics mode. `mixed_population` records provenance per genome and
`population_stats` checks the spread before you spend evaluations.


In [ ]:
seeds = lhs_population(space, 4, rng=0)  # Latin hypercube, in bounds
pop = mixed_population(
    space,
    8,
    seeds=seeds,
    fractions={"seed": 0.5, "interp": 0.25, "random": 0.25},
    rng=0,
)
print("genomes shape:", pop.genomes.shape)
print("provenance   :", pop.provenance[:3])
print(
    "stats        :",
    {k: (round(v, 4) if isinstance(v, float) else v) for k, v in population_stats(pop.genomes, space).items()},
)

## 4. The loss every optimizer sees

All three libraries **minimize**. Physics mode is exactly
`evaluator.to_minimizer()`; the `score()` wrapper below returns the loss *and* an
`EvalRecord` for the run log, without running the simulation twice.

The surrogate is target-design matching in the normalized view. It is *not*
physics — it exists so the loops converge in milliseconds on CPU — but it uses
the same space, bounds, repair and coordinates as the real objective, so the
wiring you copy is identical.


In [ ]:
# A partial fold in the normalized [-1, 1] view is the surrogate's optimum.
target_z = np.array([0.6, -0.6, 0.6, -0.6])[: space.dim]
TARGET = space.from_normalized(target_z)
target_norm = space.to_normalized(TARGET)


def surrogate_minimizer(genome):
    """Cheap stand-in: mean squared distance to the target design."""
    delta = space.to_normalized(np.asarray(genome, dtype=float)) - target_norm
    return float(np.mean(delta**2))


def score(genome):
    """Return (value to minimize, EvalRecord for RunLogger)."""
    if OBJECTIVE_MODE == "physics":
        record = evaluator.evaluate(genome)  # one real simulation
        return -record.fitness, record
    value = surrogate_minimizer(genome)
    return value, EvalRecord(fitness=-value, raw=-value, status="ok", path="surrogate")


print("objective(base_vector)   =", round(score(space.base_vector())[0], 6))
print("objective(target genome) =", round(score(TARGET)[0], 6))

## 5a. `nevergrad`

An explicit `ask()` / `tell()` loop keeps a `RunLogger` record for every
candidate. In physics mode `score()` is one full simulation per candidate.


In [ ]:
RESULTS = {}  # name -> (best genome, best loss)

import nevergrad as ng

parametrization = ng.p.Array(init=space.base_vector()).set_bounds(space.lower, space.upper)
optimizer = ng.optimizers.OnePlusOne(parametrization=parametrization, budget=BUDGET)

best_genome, best_value = None, np.inf
with RunLogger("runs/nevergrad.jsonl") as log:
    for step in range(BUDGET):
        candidate = optimizer.ask()
        genome = space.repair(np.asarray(candidate.value, dtype=float))
        value, record = score(genome)
        optimizer.tell(candidate, value)  # nevergrad minimizes
        log.log(record, genome)
        if value < best_value:
            best_genome, best_value = genome, value
        print(f"  step {step + 1}: loss={value:+.6f}")

RESULTS["nevergrad.OnePlusOne"] = (best_genome, best_value)
print(f"nevergrad best loss={best_value:+.6f}")

## 5b. CMA-ES (`cma`)

`cma` also minimizes. Bounds come from the space, `space.repair` keeps every
candidate inside them, and the loop stops at `BUDGET` evaluations. This is the
strongest baseline for the 15-variable angle space.


In [ ]:
import cma

es = cma.CMAEvolutionStrategy(
    space.base_vector().tolist(),
    0.25,
    {
        "bounds": [space.lower.tolist(), space.upper.tolist()],
        "maxfevals": BUDGET,
        "verbose": -9,
    },
)

best_genome, best_value = None, np.inf
with RunLogger("runs/cma.jsonl") as log:
    while not es.stop() and es.countevals < BUDGET:
        candidates = es.ask()
        genomes = [space.repair(np.asarray(x, dtype=float)) for x in candidates]
        values = []
        for genome in genomes:
            value, record = score(genome)
            log.log(record, genome)
            values.append(value)
        es.tell(candidates, values)
        i = int(np.argmin(values))
        if values[i] < best_value:
            best_genome, best_value = genomes[i], values[i]
        print(f"  evals={es.countevals:3d}  generation best={min(values):+.6f}")

RESULTS["cma.CMAEvolutionStrategy"] = (best_genome, best_value)
print(f"cma best loss={best_value:+.6f}")

## 5c. `pymoo`

`to_pymoo_problem(evaluator)` wraps the **real** objective as a single-objective
`pymoo` problem with the space's `xl` / `xu` bounds (returning `-fitness`,
because `pymoo` minimizes) — that is the one-liner for physics mode. Here a tiny
`Problem` subclass delegates to the same `score()`, so the demo also runs on
CPU.


In [ ]:
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.core.problem import Problem
from pymoo.optimize import minimize as pymoo_minimize

from origami_engine_mjx.integrations.adapters import to_pymoo_problem

# The adapter is constructed for free (no simulation happens here).
adapter_n_var = to_pymoo_problem(evaluator).n_var
print(f"to_pymoo_problem: n_var={adapter_n_var}")


class NotebookProblem(Problem):
    def __init__(self):
        super().__init__(n_var=space.dim, n_obj=1, xl=space.lower, xu=space.upper)

    def _evaluate(self, X, out, *args, **kwargs):
        values = []
        for row in np.asarray(X, dtype=float):
            genome = space.repair(row)
            value, _ = score(genome)
            values.append(value)
        out["F"] = np.asarray(values, dtype=float)


result = pymoo_minimize(NotebookProblem(), GA(pop_size=POP_SIZE), ("n_gen", 2), seed=0, verbose=False)
best_genome = space.repair(np.asarray(result.X, dtype=float))
best_value = float(np.ravel(result.F)[0])
RESULTS["pymoo.GA"] = (best_genome, best_value)
print(f"pymoo best loss={best_value:+.6f}")

## 6. Persist, resume, and score the winner with the physics

A checkpoint is a **normal crease-pattern JSON** (loadable by any crease tool)
plus `fitness` / `meta` keys. `load_best` returns the genome in the same space,
so a later run can warm-start from it.


In [ ]:
name = min(RESULTS, key=lambda k: RESULTS[k][1])  # every objective minimizes
best_genome, best_value = RESULTS[name]

# Score the discovered design with the real physics objective. In physics mode
# this is the optimizing objective; in surrogate mode it is a wiring check, not
# the value the optimizer saw.
record = evaluator.evaluate(best_genome)

path = save_best(
    best_genome,
    space,
    "runs/best.json",
    fitness=record.fitness,
    extra={"optimizer": name, "objective_mode": OBJECTIVE_MODE},
)
restored, meta = load_best(path, space)
assert np.allclose(restored, best_genome)

crease_path = save_genome(best_genome, space, "runs/best_pattern.json")
print(f"winner        : {name}  (loss {best_value:+.6f})")
print(f"physics score : fitness={record.fitness:+.6f} m  status={record.status}")
print(f"checkpoint    : {path}")
print(f"meta          : {meta}")
print(f"crease file   : {crease_path}")
print("best genome   :", np.round(space.repair(best_genome), 4).tolist())

## Recap and pitfalls

| Trait | Rule |
|---|---|
| Direction | `nevergrad`, `cma`, `pymoo` **minimize** → use `to_minimizer()` / `to_pymoo_problem`; `OrigamiEvaluator.fitness` is **greater-is-better** |
| Genome bounds | Always `space.repair(...)` a candidate before evaluating or decoding it |
| Normalized space | `space.to_normalized` / `space.from_normalized` give a scale-free `[-1, 1]` view for step-size control |
| Batch cost | `evaluate_batch` loops per individual; it is a *bookkeeping* path, not a speedup |
| Determinism | physics is deterministic; only sampling needs a seed (`rng=` / `np.random.default_rng`) |
| Checkpoints | `save_best` / `load_best`; the file is still a plain crease pattern |
| CPU vs GPU | every physics call is one full simulation — tiny patterns + short `duration` on CPU, the 4×4 pattern + a GPU for real budgets |

Next notebooks: **02** (every `evosax` algorithm), **03** (RL environments),
**04** (an AI / LLM agent in the loop).

Docs: `docs/OPTIMIZATION_ES.md` · `docs/SEARCH_SPACE.md` ·
`docs/POPULATIONS.md` · `docs/INTEGRATIONS.md`. The reusable integration recipe
is in `prompts/origami-optimizer-integration.md`.
